In [1]:
from graph_transformer_long_range_niches.tl import pad_batch
from graph_transformer_long_range_niches._paths import CFG_FILES, RESULTS, FIG_PATH, LEGNINI23
from graph_transformer_long_range_niches.model import LitGNNTransformer, LitGNNTransformerMasked
from graph_transformer_long_range_niches.modules import LitGCNMasked, LitGCN
from graph_transformer_long_range_niches.pl import CustomColormap, plot_attention_sender_receiver, plot_attention_sender_receiver, SelfAttentionRelevance, calculate_attention, normalized_attention, normalized_class_attention
from graph_transformer_long_range_niches.pp import prepare_geome_dataset, split_adata
from graph_transformer_long_range_niches.config import load_config
from graph_transformer_long_range_niches.tl.geome_dataloader import GraphAnnDataModule

import warnings
warnings.filterwarnings("ignore")

from torch_geometric.loader import DataLoader

from pathlib import Path
import wandb
import torch
import os

## plotting
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import pickle

import scanpy as sc
import squidpy as sq

from sklearn.preprocessing import MinMaxScaler

/ictstr01/groups/ml01/workspace/francesca.drummer/mamba/envs/GT_long_range_env/lib/python3.11/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


In [2]:
Y = torch.tensor([True, False, True, True, False])
idx = torch.where(Y)[0]
idx

tensor([0, 2, 3])

In [3]:
import random
other_idx_selected = random.sample(idx.tolist(), 2)
other_idx_selected

[3, 0]

# InterScale training


0. Create `config.yaml` file
1. Load `cfg` and `adata` object
2. Prepare PyG data
3. Train GNN (Pre-training)
4. Train Transformer (Fine-tuning, global interactions)

In [4]:
FOLDER = "legnini23"
cfg_path = "/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/Legnini_23/legnini23_InterScale_genes_sample.yaml"

### Load config and adata

In [5]:
cfg = load_config(cfg_path)

In [6]:
cfg.dataset.h5ad_data

'/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/data/legnini23.h5ad'

In [7]:
adata = sc.read_h5ad(cfg.dataset.h5ad_data)
adata

AnnData object with n_obs × n_vars = 43762 × 88
    obs: 'Cell', 'Area', 'x', 'y', 'sample', 'condition', 'organoid', 'obs_names'
    var: 'gene_ids', 'feature_types'
    obsm: 'spatial'
    layers: 'log1p_norm', 'norm_ftsqrt', 'raw'

In [8]:
adata.X = adata.layers['norm_ftsqrt']

In [9]:
# clustering

In [10]:
train_size, val_size, test_size = float(cfg.dataset.train_size), float(cfg.dataset.val_size), float(cfg.dataset.test_size)
print(f'Train size: {train_size}, val size: {val_size} and test size: {test_size}')

Train size: 0.7, val size: 0.2 and test size: 0.1


In [11]:
split_adata(adata, split_obs=cfg.dataset.obs_split, val_size=val_size, test_size=test_size, seed = cfg.optim.seed, stratify_groups = cfg.dataset.prediction_obs)

Test size > 0
{'counts': {'train': 23890, 'val': 13228, 'test': 6644}, 'groups': {'train': ['slide1_C2-1', 'slide1_C2-3', 'slide1_C2-5', 'slide4_A2-2', 'slide1_B2-3', 'slide1_A2-1', 'slide4_B2-3', 'slide4_A2-1', 'slide4_A2-3', 'slide1_A2-2', 'slide1_B2-2'], 'val': ['slide4_B2-2', 'slide1_C2-2', 'slide1_D2-3', 'slide1_D2-2'], 'test': ['slide4_B2-1', 'slide1_B2-1']}}


AnnData object with n_obs × n_vars = 43762 × 88
    obs: 'Cell', 'Area', 'x', 'y', 'sample', 'condition', 'organoid', 'obs_names', 'split'
    var: 'gene_ids', 'feature_types'
    obsm: 'spatial'
    layers: 'log1p_norm', 'norm_ftsqrt', 'raw'

In [12]:
# cross-gene vs cross-cell evaluation
cfg.set_new_allowed(True)
cfg.defrost()
cfg.optim.cross_corr = 'gene'
cfg.dataset.batch_size = 3
cfg.dataset.pct_mask_nodes = 0.5
cfg.model.decoder.type = 'linear'
#cfg.dataset.spatial_neigbors_kwargs.radius = 0
cfg.freeze()

In [13]:
pyg_data_list, _ = prepare_geome_dataset(adata, cfg)
pyg_data_list

call new
call new
call new
call new


[[Data(x=[3023, 88], edge_index=[2, 30734], obs_names=[3023]),
  Data(x=[1694, 88], edge_index=[2, 19178], obs_names=[1694]),
  Data(x=[1701, 88], edge_index=[2, 19342], obs_names=[1701]),
  Data(x=[2241, 88], edge_index=[2, 21678], obs_names=[2241]),
  Data(x=[2870, 88], edge_index=[2, 31620], obs_names=[2870]),
  Data(x=[1640, 88], edge_index=[2, 18170], obs_names=[1640]),
  Data(x=[1958, 88], edge_index=[2, 21830], obs_names=[1958]),
  Data(x=[2630, 88], edge_index=[2, 22200], obs_names=[2630]),
  Data(x=[2506, 88], edge_index=[2, 22690], obs_names=[2506]),
  Data(x=[1984, 88], edge_index=[2, 16736], obs_names=[1984]),
  Data(x=[1643, 88], edge_index=[2, 13926], obs_names=[1643])],
 [Data(x=[3510, 88], edge_index=[2, 40186], obs_names=[3510]),
  Data(x=[4318, 88], edge_index=[2, 48666], obs_names=[4318]),
  Data(x=[3621, 88], edge_index=[2, 39762], obs_names=[3621]),
  Data(x=[1779, 88], edge_index=[2, 13146], obs_names=[1779])],
 [Data(x=[3299, 88], edge_index=[2, 36290], obs_names

In [14]:
train_ds, val_ds, test_ds = pyg_data_list[0], pyg_data_list[1], pyg_data_list[2]

In [15]:
# set number of classes and number of features
cfg.dataset.merge_from_list(['num_features', len(train_ds[0].x[1])])
if 'classification' in cfg.dataset.prediction_task:
  cfg.dataset.merge_from_list(['num_classes', len(train_ds[0].y[1])])

In [16]:
dm = GraphAnnDataModule(datas=pyg_data_list, 
                           num_workers=1, 
                           batch_size=int(cfg.dataset.batch_size), 
                           pct_mask_nodes=cfg.dataset.pct_mask_nodes,
                           learning_type="node")

Masked dataloader


In [17]:
batch_mask = torch.tensor([True, True, False, False, True])
node_mask = torch.tensor([True, True, True, False, False])
len(batch_mask[node_mask])

3

In [18]:
# dm.setup("fit")
# for i, batch in enumerate(dm.train_dataloader()):
#     print(batch, np.unique(batch.batch))
#     h_node = torch.randn(batch.batch.shape[0])
#     h_node = h_node.view(-1, 2)  # Ensure it's (num_nodes, feature_dim)
#     keep_indices = batch.mask
#     print(len(keep_indices))
#     _, mask_idx = pad_batch(h_node, batch.batch, 2000, get_mask=True, keep_indices= keep_indices)

In [19]:
model = LitGNNTransformerMasked(cfg)

cross-gene per cellcorrelation metrics
cross-gene correlation metrics


In [20]:
import math
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping, Callback, Timer
import pytorch_lightning as pl

class MetricsHistory(Callback):
    def __init__(self):
        super().__init__()
        self.history = []
        
    def on_train_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        # Convert tensor values to float
        epoch_dict = {k: v.item() if hasattr(v, 'item') else v 
                     for k, v in metrics.items()}
        epoch_dict['epoch'] = trainer.current_epoch
        self.history.append(epoch_dict)

steps_per_epoch = math.ceil(len(train_ds) / cfg.dataset.batch_size)
early_stop_callback = EarlyStopping(monitor="val_r2", min_delta=0.05, patience=10*steps_per_epoch, verbose=False, mode="max")
lr_monitor = LearningRateMonitor(logging_interval='epoch')
timer = Timer()
history_callback = MetricsHistory()

In [21]:
trainer = pl.Trainer(min_epochs=1, 
                     max_epochs=3000,
                     enable_progress_bar=False,
                     callbacks=[lr_monitor, early_stop_callback, timer, history_callback],
                     log_every_n_steps=steps_per_epoch,
                     # Sanity checks: Debugging model
                     #overfit_batches=1,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
trainer.fit(model, dm)
trainer.validate(model, dm)


  | Name                   | Type                       | Params | Mode 
------------------------------------------------------------------------------
0 | loss                   | GaussianNLLLoss            | 0      | train
1 | graph_pred_linear_list | ModuleList                 | 0      | train
2 | graph_pred_linear      | Linear                     | 1.5 K  | train
3 | norm_input             | LayerNorm                  | 32     | train
4 | gnn                    | LitGCN                     | 28.4 K | train
5 | transformer_encoder    | TransformerNodeEncoderHook | 19.3 K | train
  | other params           | n/a                        | 16     | n/a  
------------------------------------------------------------------------------
49.2 K    Trainable params
0         Non-trainable params
49.2 K    Total params
0.197     Total estimated model params size (MB)
46        Modules in train mode
0         Modules in eval mode


In [ ]:
# After training, get the time metrics
total_time = timer.time_elapsed("train")  # total training time in seconds
avg_epoch_time = timer.time_elapsed("train") / trainer.current_epoch  # average time per epoch

print(f"Total training time: {total_time:.2f} seconds")
print(f"Average epoch time: {avg_epoch_time:.2f} seconds")

In [ ]:
def plot_history(history_callback, subset_term=None):
    history = pd.DataFrame(history_callback.history)
    if subset_term is not None:
        history = history[[c for c in history.columns if subset_term in c]]
    ax = history[[c for c in history.columns if 'train' in c]].plot()
    plt.gca().set_prop_cycle(None)
    history[[c for c in history.columns if 'val' in c]].plot(style='--', ax=ax)
    plt.grid(True)
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

plot_history(history_callback, subset_term='mse')
plot_history(history_callback, subset_term='r2')
plot_history(history_callback, subset_term='pearson_corr')

### Test dataloader

In [ ]:
dm.setup("fit")

In [ ]:
# dm.setup("fit")
# for i, batch in enumerate(dm.train_dataloader()):
#     print(batch, np.unique(batch.batch))
#     _, mask_idx = apply_mask(batch)

## Model evaluation

In [ ]:
# samples in test set 
adata[adata.obs['split'] == 'test'].obs.groupby(['split', 'sample', 'condition']).size()

In [27]:
dm.setup('test')

### GNNTransformer

In [28]:
def evaluate_batch(model_transformer, batch):
    self_attention_relevance = SelfAttentionRelevance(model_transformer.transformer_encoder)
    
    attention_matrix_dict = {}
    transformer_in_dict = {}
    transformer_out_dict = {}
    
    print(batch)
    transformer_in, transformer_out, src_padding_mask, index_nodes, dec_out = model_transformer.evaluation(batch)
    I = self_attention_relevance.generate_relevance(transformer_in, src_padding_mask)
    cls = I[:1, 1:].cpu().detach().numpy() 
    attention_matrix_df = pd.DataFrame(
        I[1:, 1:].cpu().detach().numpy(),
        index=sub_adata.obs_names[index_nodes[0]],
        columns=sub_adata.obs_names[index_nodes[0]]
    )
    attention_matrix_dict[str(library_id)] = attention_matrix_df
    transformer_in_dict[str(library_id)] = transformer_in
    transformer_out_dict[str(library_id)] = transformer_out

In [31]:
for i, batch in enumerate(dm.test_dataloader()):
    print(batch)

DataBatch(x=[6644, 88], edge_index=[2, 63920], obs_names=[6644], mask=[6644], batch=[6644], ptr=[3])


In [32]:
print(adata.obs_names)
print(batch.obs_names)
batch_obs_names_str = batch.obs_names.numpy().astype(int).astype(str)
adata.obs_names.isin(batch_obs_names_str).sum()

Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
       ...
       '43752', '43753', '43754', '43755', '43756', '43757', '43758', '43759',
       '43760', '43761'],
      dtype='object', length=43762)
tensor([ 4717.,  4718.,  4719.,  ..., 40337., 40338., 40339.])


6644

In [33]:
for i, batch in enumerate(dm.val_dataloader()):
    print(batch)
    for batch_idx in np.unique(batch.batch):
        transformer_in, transformer_out, src_padding_mask, index_nodes, dec_out = model.evaluation(batch)
        print(batch.obs_names[batch.batch == batch_idx])
        batch_obs_names_str = batch.obs_names[batch.batch == batch_idx].numpy().astype(int).astype(str)[index_nodes[0]]
        adata.obs_names.isin(batch_obs_names_str).sum()
        sub_adata = adata[adata.obs_names.isin(batch_obs_names_str)]
        sub_adata.layers['y_rec'] = dec_out[2000*i:2000*(i+1)].detach().numpy()
        sq.pl.spatial_scatter(
            sub_adata,
            color = ['SHH', 'PAX6', 'DBX1'],
            layer = 'norm_ftsqrt',
            cmap = 'viridis_r',
            shape= None
        )
        sq.pl.spatial_scatter(
            sub_adata,
            color = ['SHH', 'PAX6', 'DBX1'],
            layer = 'y_rec',
            cmap = 'viridis_r',
            shape= None
        )

DataBatch(x=[9607, 88], edge_index=[2, 101998], obs_names=[9607], mask=[9607], batch=[9607], ptr=[4])


TypeError: 'NoneType' object is not subscriptable

In [34]:
sub_adata = adata[adata.obs_names[index_nodes[1]]]

NameError: name 'index_nodes' is not defined

In [65]:
sub_adata.layers['y_rec'] = dec_out[2000:4000].detach().numpy()

### GCN

In [48]:
for i, batch in enumerate(dm.test_dataloader()):
    print(batch)
    x, z = model.evaluation(batch)

DataBatch(x=[6644, 88], edge_index=[2, 63920], obs_names=[6644], mask=[6644], batch=[6644], ptr=[3])


ValueError: too many values to unpack (expected 2)

In [ ]:
print(x.shape, z.shape)

In [ ]:
x_list, z_list = [], []
for batch_idx in np.unique(batch.batch):
    x_list.append(x[batch.batch == batch_idx])
    z_list.append(z[batch.batch == batch_idx])

In [ ]:
print(x_list[0].shape, x_list[1].shape)

In [ ]:
sample = 'slide1_B2-1'
idx = 1

sub_adata = adata[adata.obs['sample']==sample]
print(sub_adata)
sub_adata.layers['gnn_z'] = z_list[idx].detach().numpy()
sq.pl.spatial_scatter(
    sub_adata,
    color = ['SHH'],
    layer = 'norm_ftsqrt',
    cmap = 'viridis_r',
    shape= None
)
sq.pl.spatial_scatter(
    sub_adata,
    color = ['SHH'],
    layer = 'gnn_z',
    cmap = 'viridis_r',
    shape= None
)